In [5]:
import os
import time
import heapq
import numpy as np
import pandas as pd
from datetime import datetime
from collections import defaultdict

def log_msg(msg, logfile=None):
    timestamp = datetime.now().strftime("[%Y-%m-%d %H:%M:%S]")
    full_msg = f"[[[{timestamp}]]] {msg}"
    print(full_msg)
    if logfile:
        os.makedirs(os.path.dirname(logfile), exist_ok=True)
        with open(logfile, "a", encoding="utf-8") as f:
            f.write(full_msg + "\n")

def read_graph_dimacs_agg(file_path: str):
    agg = defaultdict(float)
    node_ids = set()

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(("c", "p")):
                continue
            if not line.startswith("a"):
                continue

            parts = line.split()
            if len(parts) < 4:
                continue

            u = parts[1]
            v = parts[2]
            try:
                w = float(parts[3])
            except ValueError:
                continue

            node_ids.add(u)
            node_ids.add(v)
            agg[(u, v)] += w

    node_list = sorted(node_ids)
    node_to_index = {node: i for i, node in enumerate(node_list)}
    index_to_node = {i: node for node, i in node_to_index.items()}

    edges_indexed = [(node_to_index[u], node_to_index[v], float(w_sum))
                     for (u, v), w_sum in agg.items()]
    edges_indexed.sort(key=lambda e: (e[0], e[1]))
    return edges_indexed, node_to_index, index_to_node

def compute_forward_backward_stats_int(edges_u, edges_v, edges_w, rank_pos):
    total = float(edges_w.sum())
    if total <= 0:
        return 0.0, 0.0, 0.0, 0.0

    fw = 0.0
    bw = 0.0
    for u, v, w in zip(edges_u, edges_v, edges_w):
        if rank_pos[u] < rank_pos[v]:
            fw += w
        else:
            bw += w
    return fw, bw, total, (fw / total)

def output_prefix_from_d_path(d_path: str) -> str:
    """
    output prefix = <same directory as .d>/<basename without .d>_graph
    Example: ../datasets/connectome.d -> ../datasets/connectome_graph
    """
    d_path = os.path.normpath(d_path)
    base = os.path.basename(d_path)
    if base.lower().endswith(".d"):
        stem = base[:-2]
    else:
        stem = os.path.splitext(base)[0]
    out_dir = os.path.dirname(d_path)
    return os.path.join(out_dir, f"{stem}_graph")

def run_adaptive_out_over_inplus1_dimacs(d_path, output_prefix=None, log_every=100000):
    t_all0 = time.perf_counter()

    if output_prefix is None:
        output_prefix = output_prefix_from_d_path(d_path)

    log_file = output_prefix + "_log.txt"
    output_csv = output_prefix + "_ranking.csv"

    try:
        # ---------------------------
        # Phase 1: Read .d and map
        # ---------------------------
        t0 = time.perf_counter()
        log_msg(f"🔹 Reading DIMACS .d file: {d_path}", log_file)

        edges_indexed, node_to_index, index_to_node = read_graph_dimacs_agg(d_path)
        n = len(node_to_index)
        m = len(edges_indexed)

        log_msg(f"✅ Graph has {n} nodes and {m} aggregated edges.", log_file)
        log_msg(f"⏱️ Read+aggregate time: {time.perf_counter() - t0:.2f}s", log_file)

        # Convert edges to numpy arrays for fast metric computation
        t1 = time.perf_counter()
        edges_u = np.fromiter((u for u, _, _ in edges_indexed), dtype=np.int32, count=m)
        edges_v = np.fromiter((v for _, v, _ in edges_indexed), dtype=np.int32, count=m)
        edges_w = np.fromiter((w for _, _, w in edges_indexed), dtype=np.float64, count=m)
        log_msg(f"⏱️ Edge array build time: {time.perf_counter() - t1:.2f}s", log_file)

        # -----------------------------------
        # Phase 2: Build adjacency + weights
        # -----------------------------------
        t2 = time.perf_counter()

        out_edges = [[] for _ in range(n)]
        in_edges  = [[] for _ in range(n)]
        out_w = np.zeros(n, dtype=np.float64)
        in_w  = np.zeros(n, dtype=np.float64)

        for u, v, w in edges_indexed:
            out_edges[u].append((v, w))
            in_edges[v].append((u, w))
            out_w[u] += w
            in_w[v] += w

        log_msg(f"⏱️ Build adjacency time: {time.perf_counter() - t2:.2f}s", log_file)

        # ---------------------------
        # Phase 3: Greedy ranking
        # ---------------------------
        t3 = time.perf_counter()

        score = (out_w + 1.0) / (in_w + 1.0)
        heap = [(-float(score[i]), i) for i in range(n)]
        heapq.heapify(heap)

        unranked = np.ones(n, dtype=bool)
        ranking_idx = np.empty(n, dtype=np.int32)
        ranked_count = 0

        while heap and ranked_count < n:
            neg_s, u = heapq.heappop(heap)
            if not unranked[u]:
                continue

            unranked[u] = False
            ranking_idx[ranked_count] = u
            ranked_count += 1

            if log_every and (ranked_count % log_every == 0):
                log_msg(f"🧩 {ranked_count} nodes ranked — last: {index_to_node[u]}", log_file)

            for v, w in out_edges[u]:
                if unranked[v]:
                    in_w[v] -= w
                    new_s = (out_w[v] + 1.0) / (in_w[v] + 1.0)
                    score[v] = new_s
                    heapq.heappush(heap, (-float(new_s), v))

            for v, w in in_edges[u]:
                if unranked[v]:
                    out_w[v] -= w
                    new_s = (out_w[v] + 1.0) / (in_w[v] + 1.0)
                    score[v] = new_s
                    heapq.heappush(heap, (-float(new_s), v))

        log_msg(f"✅ Finished ranking {ranked_count} nodes.", log_file)
        log_msg(f"⏱️ Ranking loop time: {time.perf_counter() - t3:.2f}s", log_file)

        # ---------------------------
        # Phase 4: Save + evaluate
        # ---------------------------
        t4 = time.perf_counter()

        ranking = [index_to_node[int(i)] for i in ranking_idx.tolist()]
        df_out = pd.DataFrame({"Node ID": ranking, "Order": np.arange(n, dtype=np.int64)})
        os.makedirs(os.path.dirname(output_csv) or ".", exist_ok=True)
        df_out.to_csv(output_csv, index=False)
        log_msg(f"💾 Ranking saved to: {output_csv}", log_file)

        rank_pos = np.empty(n, dtype=np.int32)
        rank_pos[ranking_idx] = np.arange(n, dtype=np.int32)

        fw, bw, total_w, ratio = compute_forward_backward_stats_int(edges_u, edges_v, edges_w, rank_pos)

        print(f"\n📊 Forward weight sum:   {fw:.6f}")
        print(f"📊 Backward weight sum:  {bw:.6f}")
        print(f"📊 Total weight sum:     {total_w:.6f}")
        print(f"📈 Forward edge ratio:   {ratio:.6f}  ({ratio*100:.2f}%)")

        log_msg(f"📊 Forward weight sum:   {fw:.6f}", log_file)
        log_msg(f"📊 Backward weight sum:  {bw:.6f}", log_file)
        log_msg(f"📊 Total weight sum:     {total_w:.6f}", log_file)
        log_msg(f"📈 Forward edge ratio:   {ratio:.6f}  ({ratio*100:.2f}%)", log_file)

        log_msg(f"⏱️ Save+metrics time: {time.perf_counter() - t4:.2f}s", log_file)

    except Exception as e:
        log_msg("❌ Error:", log_file)
        log_msg(str(e), log_file)
        raise

    finally:
        total_s = time.perf_counter() - t_all0
        print(f"\n⏱️ TOTAL RUN TIME: {total_s:.2f} seconds")
        try:
            log_msg(f"⏱️ TOTAL RUN TIME: {total_s:.2f} seconds", log_file)
        except Exception:
            pass

# === RUN ===
d_path = "../datasets/ecc.d"
run_adaptive_out_over_inplus1_dimacs(d_path)  # output prefix auto: ../datasets/connectome_graph


[[[[2026-02-08 17:21:03]]]] 🔹 Reading DIMACS .d file: ../datasets/ecc.d
[[[[2026-02-08 17:21:03]]]] ✅ Graph has 1618 nodes and 2843 aggregated edges.
[[[[2026-02-08 17:21:03]]]] ⏱️ Read+aggregate time: 0.02s
[[[[2026-02-08 17:21:03]]]] ⏱️ Edge array build time: 0.00s
[[[[2026-02-08 17:21:03]]]] ⏱️ Build adjacency time: 0.00s
[[[[2026-02-08 17:21:03]]]] ✅ Finished ranking 1618 nodes.
[[[[2026-02-08 17:21:03]]]] ⏱️ Ranking loop time: 0.00s
[[[[2026-02-08 17:21:03]]]] 💾 Ranking saved to: ../datasets/ecc_graph_ranking.csv

📊 Forward weight sum:   4089374.000000
📊 Backward weight sum:  173176.000000
📊 Total weight sum:     4262550.000000
📈 Forward edge ratio:   0.959373  (95.94%)
[[[[2026-02-08 17:21:03]]]] 📊 Forward weight sum:   4089374.000000
[[[[2026-02-08 17:21:03]]]] 📊 Backward weight sum:  173176.000000
[[[[2026-02-08 17:21:03]]]] 📊 Total weight sum:     4262550.000000
[[[[2026-02-08 17:21:03]]]] 📈 Forward edge ratio:   0.959373  (95.94%)
[[[[2026-02-08 17:21:03]]]] ⏱️ Save+metrics t

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from collections import defaultdict, Counter
import heapq

def log_msg(msg, logfile=None):
    timestamp = datetime.now().strftime("[%Y-%m-%d %H:%M:%S]")
    full_msg = f"[[[{timestamp}]]] {msg}"
    print(full_msg)
    if logfile:
        os.makedirs(os.path.dirname(logfile), exist_ok=True)
        with open(logfile, "a") as f:
            f.write(full_msg + "\n")


def parse_dimacs(filepath):
    """
    Parse a DIMACS-format directed graph file.
    Returns:
      edges: list of (u, v, weight) with weight=1 and no duplicate directed edges
      nodes: set of node IDs (as strings)
    Also prints:
      - Number of self-loops
      - Number of duplicate directed edges ignored
    """
    edges = []
    nodes = set()
    seen = set()
    duplicate_count = 0
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('c'):
                continue
            parts = line.split()
            if parts[0] == 'p':
                continue
            if parts[0] == 'a':
                _, u, v, *_ = parts
                u_str, v_str = str(u), str(v)
                if (u_str, v_str) in seen:
                    duplicate_count += 1
                    continue
                seen.add((u_str, v_str))
                nodes.update([u_str, v_str])
                edges.append((u_str, v_str, 1.0))

    loop_count = sum(1 for u, v, _ in edges if u == v)
    print(f"🔄 Number of self-loops: {loop_count}")
    print(f"⚠️ Number of duplicate directed edges ignored: {duplicate_count}")
    return edges, nodes


def run_adaptive_dimacs(input_path, output_prefix="ranking-adaptive-ibm01"):
    log_file = output_prefix + "_log.txt"
    output_csv = output_prefix + ".csv"

    try:
        log_msg(f"🔹 Parsing DIMACS graph: {input_path}", log_file)
        edges, nodes = parse_dimacs(input_path)
        log_msg(f"✅ Parsed graph with {len(nodes)} nodes and {len(edges)} edges.", log_file)

        # Build adjacency and weights
        out_edges = defaultdict(list)
        in_edges = defaultdict(list)
        out_w = defaultdict(float)
        in_w = defaultdict(float)
        for u, v, w in edges:
            out_edges[u].append(v)
            in_edges[v].append(u)
            out_w[u] += w
            in_w[v] += w

        # Initial score-based greedy ranking
        score = {u: (out_w[u] + 1) / (in_w[u] + 1) for u in nodes}
        heap = [(-score[u], u) for u in nodes]
        heapq.heapify(heap)
        unranked = set(nodes)
        ranking = []

        while heap:
            _, u = heapq.heappop(heap)
            if u not in unranked:
                continue
            ranking.append(u)
            unranked.remove(u)
            for nbr in out_edges[u]:
                if nbr in unranked:
                    in_w[nbr] -= 1.0
                    score[nbr] = (out_w[nbr] + 1) / (in_w[nbr] + 1)
                    heapq.heappush(heap, (-score[nbr], nbr))
            for nbr in in_edges[u]:
                if nbr in unranked:
                    out_w[nbr] -= 1.0
                    score[nbr] = (out_w[nbr] + 1) / (in_w[nbr] + 1)
                    heapq.heappush(heap, (-score[nbr], nbr))

        if unranked:
            rem = list(unranked)
            np.random.shuffle(rem)
            ranking.extend(rem)

        log_msg(f"✅ Finished ranking {len(ranking)} nodes.", log_file)

        # Save only Node ID and Order, sorted by Order
        df_out = pd.DataFrame({
            "Node ID": ranking,
            "Order": list(range(len(ranking)))
        })
        df_out.to_csv(output_csv, index=False)
        log_msg(f"💾 Saved ranking to {output_csv}", log_file)

        # Count backward edges
        rank_map = {node: i for i, node in enumerate(ranking)}
        backward_count = sum(1 for u, v, _ in edges if rank_map[u] > rank_map[v])
        print(f"🔙 Number of backward edges: {backward_count}")
        log_msg(f"🔙 Number of backward edges: {backward_count}", log_file)

    except Exception as e:
        log_msg("❌ Error encountered:", log_file)
        log_msg(str(e), log_file)

# Example usage:
input_path = "/content/drive/MyDrive/ibm01.txt"
run_adaptive_dimacs(input_path, output_prefix="/content/drive/MyDrive/ibm01_ranking")




[[[[2025-06-11 22:41:36]]]] 🔹 Parsing DIMACS graph: /content/drive/MyDrive/ibm01.txt
🔄 Number of self-loops: 0
⚠️ Number of duplicate directed edges ignored: 633
[[[[2025-06-11 22:41:36]]]] ✅ Parsed graph with 12752 nodes and 36048 edges.
[[[[2025-06-11 22:41:36]]]] ✅ Finished ranking 12752 nodes.
[[[[2025-06-11 22:41:36]]]] 💾 Saved ranking to /content/drive/MyDrive/ibm01_ranking.csv
🔙 Number of backward edges: 3952
[[[[2025-06-11 22:41:36]]]] 🔙 Number of backward edges: 3952
